# Dataset EDA

In [1]:
import pandas as pd

recipes = pd.read_csv("../data/recipes.csv")
reviews = pd.read_csv("../data/reviews.csv")

print("Recipes shape:", recipes.shape)
print("Reviews shape:", reviews.shape)

recipes.head()

Recipes shape: (522517, 28)
Reviews shape: (1401982, 8)


,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,...,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen ...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,NaN,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_...",...,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,NaN,"c(""Soak saffron in warm milk for 5 minutes and..."
2,40,Best Lemonade,1566,Stephen Little,PT5M,PT30M,PT35M,1999-09-05T19:52:00Z,This is from one of my first Good House Keepi...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,0.0,0.0,1.8,81.5,0.4,77.2,0.3,4.0,NaN,"c(""Into a 1 quart Jar with tight fitting lid, ..."
3,41,Carina's Tofu-Vegetable Kebabs,1586,Cyclopz,PT20M,PT24H,PT24H20M,1999-09-03T14:54:00Z,This dish is best prepared a day in advance to...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,3.8,0.0,1558.6,64.2,17.3,32.1,29.3,2.0,4 kebabs,"c(""Drain the tofu, carefully squeezing out exc..."
4,42,Cabbage Soup,1538,Duckie067,PT30M,PT20M,PT50M,1999-09-19T06:19:00Z,Make and share this Cabbage Soup recipe from F...,"""https://img.sndimg.com/food/image/upload/w_55...",...,0.1,0.0,959.3,25.1,4.8,17.7,4.3,4.0,NaN,"c(""Mix everything together and bring to a boil..."


In [2]:
recipes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522517 entries, 0 to 522516
Data columns (total 28 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   RecipeId                    522517 non-null  int64  
 1   Name                        522517 non-null  object 
 2   AuthorId                    522517 non-null  int64  
 3   AuthorName                  522517 non-null  object 
 4   CookTime                    439972 non-null  object 
 5   PrepTime                    522517 non-null  object 
 6   TotalTime                   522517 non-null  object 
 7   DatePublished               522517 non-null  object 
 8   Description                 522512 non-null  object 
 9   Images                      522516 non-null  object 
 10  RecipeCategory              521766 non-null  object 
 11  Keywords                    505280 non-null  object 
 12  RecipeIngredientQuantities  522514 non-null  object 
 13  RecipeIngredie

In [3]:
recipes.isnull().sum().sort_values(ascending=False)

RecipeYield                   348071
AggregatedRating              253223
ReviewCount                   247489
RecipeServings                182911
CookTime                       82545
Keywords                       17237
RecipeCategory                   751
Description                        5
RecipeIngredientQuantities         3
Images                             1
Name                               0
AuthorId                           0
TotalTime                          0
PrepTime                           0
DatePublished                      0
AuthorName                         0
RecipeId                           0
FatContent                         0
Calories                           0
RecipeIngredientParts              0
CholesterolContent                 0
SaturatedFatContent                0
SodiumContent                      0
CarbohydrateContent                0
SugarContent                       0
FiberContent                       0
ProteinContent                     0
R

In [4]:
recipes["RecipeIngredientParts"].head()

0    c("blueberries", "granulated sugar", "vanilla ...
1    c("saffron", "milk", "hot green chili peppers"...
2    c("sugar", "lemons, rind of", "lemon, zest of"...
3    c("extra firm tofu", "eggplant", "zucchini", "...
4    c("plain tomato juice", "cabbage", "onion", "c...
Name: RecipeIngredientParts, dtype: object

In [5]:
type(recipes["RecipeIngredientParts"].iloc[0])

str

In [6]:
import ast
import re

def clean_ingredients(text):
    """
    Convert R-style ingredient string:
    c("sugar", "milk")
    → Python list: ["sugar", "milk"]
    """
    if pd.isna(text):
        return []

    # Remove leading c( and trailing )
    text = re.sub(r'^c\(|\)$', '', text)

    # Split by comma while keeping quoted words
    items = re.findall(r'"(.*?)"', text)

    # Normalize: lowercase + strip spaces
    items = [i.lower().strip() for i in items]

    return items

In [7]:
recipes["clean_ingredients"] = recipes["RecipeIngredientParts"].apply(clean_ingredients)
recipes["clean_ingredients"].head()

0    [blueberries, granulated sugar, vanilla yogurt...
1    [saffron, milk, hot green chili peppers, onion...
2    [sugar, lemons, rind of, lemon, zest of, fresh...
3    [extra firm tofu, eggplant, zucchini, mushroom...
4    [plain tomato juice, cabbage, onion, carrots, ...
Name: clean_ingredients, dtype: object

In [8]:
len(recipes["clean_ingredients"].iloc[0])

4

# test logic

#### novelty 1 - missing ingredients <= 2

In [9]:
user_ingredients = ["onion", "salt", "rice"]
user_ingredients = [i.lower().strip() for i in user_ingredients]

In [10]:
def ingredient_gap_score(recipe_ings, user_ings):
    recipe_set = set(recipe_ings)
    user_set = set(user_ings)

    have = recipe_set.intersection(user_set)
    missing = recipe_set.difference(user_set)

    return len(have), len(missing), list(missing)

In [11]:
recipes[["have_count", "missing_count", "missing_ings"]] = recipes[
    "clean_ingredients"
].apply(lambda x: pd.Series(ingredient_gap_score(x, user_ingredients)))

In [12]:
candidate_recipes = recipes[recipes["missing_count"] <= 2]

In [13]:
candidate_recipes.shape

(34107, 32)

In [14]:
candidate_recipes[["Name", "missing_count"]].head(5)

,Name,missing_count
8,A Jad - Cucumber Pickle,2
30,Chicha Peruana,2
85,Champagne Punch,2
90,Cherry Sandwich Maker Snack,1
92,Cheese and Pineapple Dip,2


#### novelty 2 - Budget-Aware Scoring

In [15]:
ingredient_cost = {
    "salt": 1,
    "sugar": 1,
    "onion": 1,
    "rice": 1,
    "oil": 1,
    "milk": 2,
    "cheese": 3,
    "chicken": 3,
    "butter": 2,
    "egg": 2
}

In [16]:
def cost_score(missing_ings):
    return sum(ingredient_cost.get(ing, 2) for ing in missing_ings)

In [17]:
candidate_recipes["missing_cost"] = candidate_recipes["missing_ings"].apply(cost_score)

C:\Users\HP\AppData\Local\Temp\ipykernel_16416\3887581741.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidate_recipes["missing_cost"] = candidate_recipes["missing_ings"].apply(cost_score)


In [18]:
candidate_recipes["final_score"] = (
    candidate_recipes["have_count"] - candidate_recipes["missing_cost"]
)

C:\Users\HP\AppData\Local\Temp\ipykernel_16416\3449196168.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidate_recipes["final_score"] = (


In [19]:
top_recipes = candidate_recipes.sort_values(
    by="final_score", ascending=False
).head(10)

top_recipes[["Name", "have_count", "missing_count", "missing_cost", "final_score"]]

,Name,have_count,missing_count,missing_cost,final_score
514698,White Rice,3,0,0,3
129792,Pea Soup,2,0,0,2
52094,Neer Dosa,2,0,0,2
499795,Roasted Oysters With Hot Sauce,1,0,0,1
237881,Southern-Fried Potatoes,1,0,0,1
231511,Guam Red Rice,3,1,2,1
164910,Easy Homemade Almond Butter,1,0,0,1
183792,Microwave Those Pumpkin Seeds !,1,0,0,1
139575,Home Butter - Homemade,1,0,0,1
149778,Roast Pork Shoulder,1,0,0,1


In [20]:
recipes.to_pickle("../data/recipes_clean.pkl")

In [21]:
candidate_recipes.to_pickle("../data/candidate_recipes.pkl")